# Cyberbullying AI Detection - Google Colab Training Notebook

This notebook trains the machine learning pipeline for the **Cyberbullying AI Safe Chat** application.
It uses TF-IDF n-gram vectorization with a calibrated Linear Support Vector Machine (LinearSVC) to predict whether incoming text constitutes cyberbullying or safe communication.

### Output Bundle:
At the end of this notebook, a single zip file `colab_model_bundle.zip` will be downloaded containing:
1. `cyberbullying_pipeline.pkl` (The trained pipeline ready for the web app)
2. `model_metadata.json` (Training statistics, accuracy, F1-score, and timestamps)
3. `reports/confusion_matrix.png` & `classification_report.txt`

## 1. Install & Import Dependencies

In [ ]:
!pip install --upgrade scikit-learn joblib pandas numpy matplotlib seaborn

import os
import re
import html
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

print("Dependencies loaded successfully!")

## 2. Load the Dataset
Upload `cyberbullying_tweets.csv` directly into Colab, or upload via the file browser on the left panel.

In [ ]:
from google.colab import files

dataset_filename = "cyberbullying_tweets.csv"
if not os.path.exists(dataset_filename):
    print(f"Please upload '{dataset_filename}'...")
    uploaded = files.upload()
else:
    print(f"Found existing '{dataset_filename}'.")

## 3. Data Inspection & Cleaning
We use the exact normalization logic as the production backend (`clean_text`) to guarantee consistency between training and real-time chat inference.

In [ ]:
URL_RE = re.compile(r"https?:\/\/\S+|www\.\S+", re.IGNORECASE)
MENTION_RE = re.compile(r"@\w+")
WHITESPACE_RE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    """Linguistic normalization used by both training and web inference."""
    if text is None:
        return ""
    text = html.unescape(str(text))
    text = text.lower().strip()
    text = URL_RE.sub(" URL ", text)
    text = MENTION_RE.sub(" USER ", text)
    text = WHITESPACE_RE.sub(" ", text)
    return text.strip()

# Load and filter CSV
df = pd.read_csv(dataset_filename)
print(f"Raw dataset rows: {len(df):,}")
print("Subtypes in dataset:\n", df["cyberbullying_type"].value_counts())

df = df[["tweet_text", "cyberbullying_type"]].dropna().copy()
df["tweet_text"] = df["tweet_text"].astype(str).str.strip()
df = df[df["tweet_text"].str.len() > 0]
df = df.drop_duplicates(subset=["tweet_text", "cyberbullying_type"])

# Binary label mapping: not_cyberbullying -> 0 (SAFE), all bullying types -> 1 (CYBERBULLYING)
df["label"] = (df["cyberbullying_type"] != "not_cyberbullying").astype(int)
print(f"\nCleaned rows: {len(df):,}")
print("Binary distribution:")
print(df["label"].value_counts().rename(index={0: "SAFE", 1: "CYBERBULLYING"}))

## 4. Train-Test Split & Pipeline Construction
We construct a scikit-learn Pipeline with:
- `TfidfVectorizer` (sublinear term frequencies, 1-2 n-grams, unicode accent stripping)
- `CalibratedClassifierCV` wrapping `LinearSVC(class_weight='balanced')` with sigmoid calibration for accurate probability confidence scores.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["tweet_text"],
    df["label"],
    test_size=0.20,
    random_state=42,
    stratify=df["label"],
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        preprocessor=clean_text,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        max_features=100_000,
        sublinear_tf=True,
        strip_accents="unicode",
    )),
    ("classifier", CalibratedClassifierCV(
        estimator=LinearSVC(C=1.0, class_weight="balanced", random_state=42),
        method="sigmoid",
        cv=5,
        n_jobs=-1,
    ))
])

print("Training pipeline on GPU/CPU...")
pipeline.fit(X_train, y_train)
print("Training completed!")

## 5. Model Evaluation & Metrics

In [ ]:
predictions = pipeline.predict(X_test)
probabilities = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)

print("=" * 40)
print(f"Accuracy : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print("=" * 40)

report = classification_report(
    y_test, 
    predictions, 
    target_names=["SAFE", "CYBERBULLYING"], 
    digits=4
)
print(report)

# Create confusion matrix plot
os.makedirs("reports", exist_ok=True)
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
disp = ConfusionMatrixDisplay(cm, display_labels=["SAFE", "CYBERBULLYING"])
disp.plot(cmap=plt.cm.Blues, values_format="d")
plt.title("Cyberbullying AI Safe Chat - Confusion Matrix")
plt.tight_layout()
plt.savefig("reports/confusion_matrix.png", dpi=200)
plt.show()

## 6. Export Model Bundle for Web App
Run this cell to save `cyberbullying_pipeline.pkl`, create `model_metadata.json`, and download `colab_model_bundle.zip`.
Once downloaded, you can drag-and-drop the file directly into the **Model Upload Hub** in the web app's Admin panel or extract it into your project folder!

In [ ]:
os.makedirs("model", exist_ok=True)
model_path = "model/cyberbullying_pipeline.pkl"
metadata_path = "model/model_metadata.json"

# Save pipeline
joblib.dump(pipeline, model_path)

# Metadata for web dashboard
metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_file": dataset_filename,
    "task": "Binary Cyberbullying Detection",
    "model": "TF-IDF (1-2 n-grams) + Calibrated LinearSVC (Sigmoid)",
    "default_block_threshold": 0.50,
    "accuracy": round(float(accuracy), 4),
    "precision": round(float(precision), 4),
    "recall": round(float(recall), 4),
    "f1": round(float(f1), 4),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "average_test_bullying_probability": round(float(probabilities.mean()), 4)
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

with open("reports/classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

# Bundle into a single zip archive for easy transfer
shutil.make_archive("colab_model_bundle", "zip", root_dir=".", base_dir="model")
print("\nBundle created: colab_model_bundle.zip")

# Download bundle directly
try:
    from google.colab import files
    files.download("model/cyberbullying_pipeline.pkl")
    files.download("model/model_metadata.json")
    files.download("colab_model_bundle.zip")
    print("Triggered downloads! Check your downloads folder.")
except Exception as e:
    print("Download trigger note:", e)
    print("You can manually download files from the left file explorer in Colab.")